# Knee-OA Pipeline — boundary soft labels (OA 0↔1 + KL1↔2) + mild medial JSN

Before running: **Settings → Accelerator → GPU T4 x2** and **Settings → Internet → On**.
Inputs: `knee-oa-data` + **updated** `knee-oa-code` (must include `--soft-oa01` and `--aux-jsn-med-weight`).

To resume: add *this notebook's previous version output* as an input.

**Do not overwrite** locked runs. Option C on attn_spatial:
`--soft-oa01 0.1` (KL0↔1) + `--soft-kl12 0.1` (KL1↔2) + `--aux-jsn-med-weight 1.5`.

**Save Version → Save & Run All (Commit)**. Download `results_knee_oa.zip` when done.


In [ ]:
# ============================ CONFIG ============================
SMOKE_TEST = False    # True: tiny 1-epoch check (~10 min). False: real training.
EPOCHS = 15
BASELINE = "r50_dual_cascade_sqrt_aux_attn_spatial"  # locked weights; cutpoints QWK 0.867
SESSION_HOURS = 11.0  # Kaggle kills sessions at 12 h; training pauses safely before this

# Option C: soft OA gate + soft KL1/2 + mild medial JSN aux
RUNS = [
    ("r50_dual_cascade_sqrt_aux_attn_spatial_boundary", [0, 1, 2, 3, 4],
     "--backbone resnet50 --head cascade --sampler sqrt "
     "--aux-csv reports/pipeline/folds_features.csv --aux-weight 0.75 "
     "--spatial-aux --attn-band-weight 0.1 --attn-border-frac 0.08 "
     "--soft-oa01 0.1 --soft-kl12 0.1 --aux-jsn-med-weight 1.5"),
]



In [ ]:
# ============================ SETUP (no edits needed) ============================
import glob, os, shutil, subprocess, sys, time
from pathlib import Path
import torch, torchvision

T_START = time.time()
os.environ["PIPELINE_DEADLINE"] = str(T_START + SESSION_HOURS * 3600)
INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
RUNS_DIR = WORK / ("smoke_runs" if SMOKE_TEST else "cv_runs")

def find_dir(name, what):
    hits = sorted(glob.glob(str(INPUT / "**" / name), recursive=True))
    if not hits:
        raise SystemExit(f"Could not find {what} under /kaggle/input. Add it with 'Add Input' (right panel).")
    return Path(hits[0])

CODE = Path("/tmp/knee_oa_code")  # not in /kaggle/working, so old code never ends up in outputs
shutil.rmtree(CODE, ignore_errors=True)
shutil.copytree(find_dir("knee_oa_code", "the code dataset (knee_oa_code)"), CODE)
DATA = find_dir("Dataset_combined", "the data dataset (Dataset_combined)")
n_imgs = len(glob.glob(str(DATA / "*" / "*" / "*.png")))
print("code:", CODE, "| data:", DATA, f"({n_imgs} images)")
assert n_imgs > 6000, "data dataset looks incomplete"

if not SMOKE_TEST:  # resume: restore prior results (incl. last.pth for unfinished folds)
    for prev in sorted(glob.glob(str(INPUT / "**" / "cv_runs"), recursive=True)):
        print("restoring previous results from", prev)
        shutil.copytree(prev, RUNS_DIR, dirs_exist_ok=True)
        n_last = len(list(RUNS_DIR.glob("*/fold*/last.pth")))
        n_done = len(list(RUNS_DIR.glob("*/fold*/DONE")))
        print(f"  restored DONE={n_done} last.pth(unfinished resume)={n_last}")

gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("GPUs:", gpus)
assert gpus, "No GPU. Settings -> Accelerator -> GPU T4 x2, then run again."

# download ImageNet weights once here (two folds starting in parallel would race on the same file)
try:
    torchvision.models.resnet50(weights="IMAGENET1K_V2")
    torchvision.models.convnext_tiny(weights="IMAGENET1K_V1")
    torchvision.models.efficientnet_v2_s(weights="IMAGENET1K_V1")
except Exception as e:
    raise SystemExit(f"Could not download pretrained weights ({e}). Turn on Settings -> Internet.")
print("setup ok")


In [ ]:
# ============================ TRAIN ============================
def run(name, folds, extra):
    cmd = [sys.executable, "-u", str(CODE / "src/training/stage4_run_folds.py"),
           "--run-name", name, "--folds", *map(str, folds), "--out-dir", str(RUNS_DIR),
           "--data-root", str(DATA), "--epochs", str(EPOCHS), "--num-workers", "2", *extra.split()]
    if SMOKE_TEST:
        cmd += ["--epochs", "1", "--max-train-batches", "20", "--max-val-batches", "10"]
    print(f"\n===== {name} | folds {folds} | {time.strftime('%H:%M:%S')} =====", flush=True)
    p = subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    return p.wait()

for name, folds, extra in RUNS:
    if time.time() > float(os.environ["PIPELINE_DEADLINE"]) - 3600:
        print(f"skipping {name}: less than 1 h left in this session (it will run next session)")
        continue
    rc = run(name, folds, extra)
    print(f"===== {name} finished with exit code {rc} after {(time.time() - T_START) / 3600:.2f} h total =====")

for log in sorted(RUNS_DIR.glob("*/fold*/train.log")):  # epoch-by-epoch history of every fold
    print(f"\n--- {log.parent.parent.name}/{log.parent.name} ---")
    print("\n".join(l for l in log.read_text().splitlines() if "epoch" in l or "done" in l or "warn" in l or "Error" in l))

In [ ]:
# ============================ OOF REPORT + PACK RESULTS ============================
cmd = [sys.executable, str(CODE / "src/training/stage4_oof_report.py"),
       "--runs-dir", str(RUNS_DIR), "--out", str(WORK / "oof_report")]
if list(RUNS_DIR.glob(f"{BASELINE}/fold*/oof.npz")):
    cmd += ["--baseline", BASELINE]
print(subprocess.run(cmd, cwd=CODE, capture_output=True, text=True).stdout or "no finished folds yet")

# one zip to download: weights + OOF + logs; keep last.pth only for unfinished folds (resume)
import zipfile
zip_path = WORK / f"results_{'smoke' if SMOKE_TEST else 'knee_oa'}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as zf:
    for f in [*RUNS_DIR.rglob("*"), *(WORK / "oof_report").rglob("*")]:
        if not f.is_file():
            continue
        if f.name.startswith("last.pth") and (f.parent / "DONE").exists():
            continue  # finished fold: drop resume blob
        zf.write(f, f.relative_to(WORK))
print(zip_path, f"{zip_path.stat().st_size / 2**20:.0f} MB")

# Kaggle's final save stalls when the same checkpoints exist twice (folder + zip).
shutil.rmtree(RUNS_DIR, ignore_errors=True)
shutil.rmtree(WORK / "oof_report", ignore_errors=True)
print("notebook code finished. Only the zip remains; Kaggle may take a few minutes to publish it.")
